# 6-tap M2 on PACS (Colab T4)

Trains the missing control from the paper: **6-tap M2 with no contrastive loss**, for 4 held-out domains and 3 seeds (12 runs of 30 epochs). It uses the same settings as corrected M2-CL (lr 5e-4, batch 128, baseline augmentation) and the fixed train/validation split.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload to the top level of **My Drive**: `M2CL_code.zip` (from `colab_upload/`) and `PACS.zip` (already there if you used your earlier notebooks).
3. Run the cells in order.

**If Colab disconnects:** reconnect and run all the cells again. Checkpoints are built in, so you don't have to set anything:
- After every epoch, `latest.pt` is saved to `My Drive/m2cl_v2/runs/<run>/`, and the previous one is kept in case the newest is damaged.
- Finished runs are skipped, and an interrupted run continues from its last saved epoch. You lose at most the epoch in progress, or slightly more if Drive hadn't finished syncing.
- The cell just before training shows where each run stands, and a resumed run prints `[resume] ... from epoch N`.

**Drive space:** finished runs need about 1 GB, plus up to about 0.3 GB for the run in progress. Each epoch replaces a large checkpoint file, and Drive may keep the old versions in its trash, so empty the trash if space runs low.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

CODE_ZIP = '/content/drive/MyDrive/M2CL_code.zip'
PACS_ZIP = '/content/drive/MyDrive/PACS.zip'
OUT      = '/content/drive/MyDrive/m2cl_v2'

import os, shutil, zipfile
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
for path in (CODE_ZIP, PACS_ZIP):
    assert os.path.exists(path), f'Not found: {path}. Upload it to the top level of My Drive.'

shutil.rmtree('/content/repo', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/repo')
os.chdir('/content/repo')
!pip -q install pyyaml tabulate

if not os.path.exists('/content/PACS_raw'):
    print('Extracting PACS...')
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')

# Use the real images: the folder of four domains whose files are not dct_* (docs/AUDIT.md, A0)
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
matches = [r for r, d, _ in os.walk('/content/PACS_raw')
           if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
assert matches, 'No RGB copy of PACS found in PACS.zip.'
DATA = matches[0]
n_images = sum(len(f) for _, _, f in os.walk(DATA))
print('PACS root:', DATA, '| images:', n_images)
assert n_images == 9991, f'Expected 9,991 images, found {n_images}.'
os.makedirs(OUT, exist_ok=True)

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f'Free Drive space: {free_gb:.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free. Empty the Drive trash before training.')

## 2. Where each run stands
Run this any time. `continues at epoch N` means the next training cell picks up from there.

In [ ]:
import json, torch
import pandas as pd

def run_status(target, seed):
    run_dir = f'{OUT}/runs/m2_6tap_{target}_seed{seed}'
    result = os.path.join(run_dir, 'result.json')
    if os.path.exists(result) and json.load(open(result)).get('status') == 'complete':
        return 'finished'
    for name in ('latest.pt', 'latest_prev.pt'):
        try:
            return f"continues at epoch {torch.load(os.path.join(run_dir, name), map_location='cpu', weights_only=False)['epoch'] + 1}"
        except Exception:
            pass
    return 'not started'

display(pd.DataFrame([{'target': t, 'seed': s, 'status': run_status(t, s)}
                      for t in ('art_painting', 'cartoon', 'photo', 'sketch') for s in (0, 1, 2)]))

## 3. Train 6-tap M2 (12 runs)
The first epoch line shows the time per epoch, so you can estimate the total: seconds per epoch × 30 × 12. Keep the tab open; free Colab may stop the session, and you can then rerun everything to continue.

In [ ]:
!python scripts/train_lodo.py --config configs/m2_6tap.yaml --data_root "$DATA" --out_dir "$OUT/runs"

## 4. Check the results
This should end with `12 of 12 runs complete`. If it doesn't, run the training cell again.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{OUT}/runs/m2_6tap_*/result.json')):
    r = json.load(open(path))
    if r.get('status') != 'complete':
        continue
    c = r['config']
    best = os.path.join(os.path.dirname(path), 'best.pt')
    assert c['model'] == 'm2' and c['arch_taps'] == 6 and c['alpha'] == 0, f'Wrong settings in {path}'
    assert os.path.exists(best), f'Missing checkpoint: {best}'
    rows.append({'target': c['target'], 'seed': c['seed'], 'test_acc': r['test_acc'], 'best_epoch': r['best_epoch'],
                 'device': r['device'], 'torch': r['torch']})
runs = pd.DataFrame(rows)
print(f'{len(runs)} of 12 runs complete')
if len(runs):
    display(runs.pivot(index='target', columns='seed', values='test_acc').round(2))
    print(runs.groupby('target').test_acc.agg(['mean', 'std']).round(2))
    print('Macro-average of the domain means:', round(runs.groupby('target').test_acc.mean().mean(), 2))
    print('Devices:', runs.device.unique(), '| torch:', runs.torch.unique())

## Next steps
- **Tell Claude the results are ready.** Download `My Drive/m2cl_v2/runs/` to the laptop, where the tap-level probes run so that they stay on the same compute as before.
- **Don't compare these runs directly with the old M2-CL numbers yet.** Those M2-CL runs used the leaky validation split, so any difference could come from the split rather than the loss. The optional cell below trains M2-CL with the fixed split, which gives a fair comparison.

## Optional: 6-tap M2-CL with the fixed split (12 more runs)
Needed before the paper can compare M2 with M2-CL. It uses the same output folder and can also resume. Set `TRAIN_M2CL = True` to run it.

In [ ]:
TRAIN_M2CL = False
if TRAIN_M2CL:
    !python scripts/train_lodo.py --config configs/m2cl_6tap.yaml --data_root "$DATA" --out_dir "$OUT/runs"